[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/13_polars-duckdb.ipynb)

*Python for Data Science · Notebook 13 of 13*

# Polars & DuckDB

The modern, fast way to wrangle data on one machine: Polars expressions, lazy queries, window functions, joins, pivots, rolling windows, strings, nulls, lists and dedupe — and DuckDB SQL straight on your data: HAVING, running totals and LAG, QUALIFY, FILTER, ASOF joins, lists and gap-filled dates. Every answer is also shown in plain Python.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** pandas is a trusty hand tool; Polars and DuckDB are power tools — same jobs, much faster, and they read only what they need.

**What interviewers really test here:** Polars expressions and lazy queries, window functions with over(), joins and anti joins, DuckDB SQL on files and DataFrames, QUALIFY, ASOF joins, and when to choose which tool.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why Polars and DuckDB? *(Basics)*
- [ ] 2. Polars expressions *(Core)*
- [ ] 3. Reshaping and windows in Polars *(Core)*
- [ ] 4. DuckDB: SQL on files and DataFrames *(Core)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### pandas → Polars → DuckDB, side by side

| Task | pandas | Polars | DuckDB SQL |
| --- | --- | --- | --- |
| Filter | `df[df.x > 5]` | `df.filter(pl.col("x") > 5)` | `WHERE x > 5` |
| New column | `df.assign(y=df.x * 2)` | `with_columns((pl.col("x") * 2).alias("y"))` | `SELECT *, x * 2 AS y` |
| Group | `groupby("g").x.sum()` | `group_by("g").agg(pl.col("x").sum())` | `GROUP BY g` |
| Window | `groupby("g").x.transform("sum")` | `pl.col("x").sum().over("g")` | `SUM(x) OVER (PARTITION BY g)` |
| Join | `merge(b, on="k", how="left")` | `join(b, on="k", how="left")` | `LEFT JOIN b USING (k)` |
| Lazy | — | `scan_parquet(...).collect()` | always planned |

---
# Part 2: Lessons, step by step

## Lesson 1: Why Polars and DuckDB?

*Level: Basics*

pandas is everywhere, but on bigger data it runs out of memory and uses one CPU core. Two newer tools fix that on a single machine, and many job posts now list them:



|   | pandas | Polars | DuckDB |
| --- | --- | --- | --- |
| Style | DataFrame methods | DataFrame **expressions** | **SQL** |
| Engine | mostly single-threaded | multi-threaded, Rust | multi-threaded, columnar, C++ |
| Lazy / optimised | no | yes (`lazy()`, `scan_*`) | yes, a query planner |
| Bigger than memory | no | streaming engine | spills to disk |
| Reads | CSV, Parquet… | CSV, Parquet, JSON, Delta | CSV, Parquet, JSON, S3, Postgres… |



They work together: DuckDB can query pandas or Polars frames by name, and both convert with `.to_pandas()`, `pl.from_pandas()` and Arrow without copying much.



**Running the code:** DuckDB runs in this site's in-browser IDE. Polars doesn't run in the browser yet — use Colab (it's preinstalled there) or `pip install polars duckdb` locally. The IDE still checks plain-Python or pandas answers to every question.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 2: Polars expressions

*Level: Core*

In Polars you describe *what* to compute with **expressions** — `pl.col("amount") * 1.18` — and hand them to `select`, `with_columns`, `filter` or `group_by().agg()`. Expressions in one call run in parallel, and there's no index to manage.



```python
import polars as pl
orders = pl.read_csv("orders.csv")

(orders
 .filter(pl.col("status") == "completed")
 .with_columns(
     (pl.col("amount") * 1.18).round(2).alias("with_tax"),
     pl.when(pl.col("amount") < 200).then(pl.lit("small"))
       .when(pl.col("amount") < 1000).then(pl.lit("medium"))
       .otherwise(pl.lit("large")).alias("band"),
 )
 .group_by("city")
 .agg(pl.len().alias("orders"), pl.col("amount").sum().alias("revenue"), pl.col("amount").mean().round(1))
 .sort("revenue", descending=True))

# pandas equivalent of the group_by:
# df[df.status == "completed"].groupby("city").agg(orders=("id", "size"), revenue=("amount", "sum"))
```



**Lazy mode**: `pl.scan_csv(...)` or `df.lazy()` builds a query plan; Polars pushes filters and column selection down to the file reader and runs only on `.collect()`. `plan.explain()` shows the optimised plan.

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q3](#q3), [Q12](#q12), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 3: Reshaping and windows in Polars

*Level: Core*

```python
# window functions: compute per group, keep every row
df.with_columns(
    pl.col("amount").rank("dense", descending=True).over("city").alias("rank_in_city"),
    (pl.col("amount") / pl.col("amount").sum().over("city")).alias("share"),
    pl.col("temp").forward_fill().over("sensor"),
)
orders.join(customers, on="customer", how="left")       # also inner, full, semi, anti
customers.join(orders, on="customer", how="anti")        # customers with no orders
sales.pivot(on="city", index="month", values="revenue")  # long → wide; unpivot() goes back
daily.with_columns(pl.col("orders").rolling_mean(window_size=3))
posts.explode("tags")                                    # one row per list item
changes.sort("updated").unique("id", keep="last")        # latest version per id
```

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q4](#q4), [Q16](#q16), [Q6](#q6), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 4: DuckDB: SQL on files and DataFrames

*Level: Core*

DuckDB is an in-process analytical database — SQLite for analytics. There's no server: `import duckdb` and query CSV or Parquet files, folders of files, S3 paths, or DataFrames in memory.



```python
import duckdb
duckdb.sql("SELECT city, SUM(amount) FROM 'orders/*.parquet' GROUP BY city").show()
duckdb.sql("SELECT * FROM orders_df WHERE amount > 1000").pl()      # query a pandas frame, get Polars back

con = duckdb.connect("analytics.duckdb")                            # or a persistent file
con.execute("CREATE TABLE orders AS SELECT * FROM read_csv('orders.csv')")
```



Its SQL has handy extras: `QUALIFY` (filter on window functions), `COUNT(*) FILTER (WHERE …)`, `ASOF JOIN` for time series, `list()` and `string_agg()`, `generate_series`, `PIVOT`, and `GROUP BY ALL`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import duckdb


def duckdb_demo():
    con = duckdb.connect()
    con.execute("CREATE TABLE sales AS SELECT * FROM (VALUES ('Pune', 120.0), ('Delhi', 90.0), ('Pune', 150.0)) t(city, revenue)")
    print(con.execute("SELECT city, SUM(revenue) AS total FROM sales GROUP BY city ORDER BY total DESC").fetchall())
    print(con.execute("SELECT city, revenue FROM sales QUALIFY ROW_NUMBER() OVER (PARTITION BY city ORDER BY revenue DESC) = 1 ORDER BY city").fetchall())


duckdb_demo()

[('Pune', Decimal('270.0')), ('Delhi', Decimal('90.0'))]
[('Delhi', Decimal('90.0')), ('Pune', Decimal('150.0'))]


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q17](#q17), [Q18](#q18), [Q9](#q9), [Q19](#q19), [Q10](#q10), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
import polars as pl

import duckdb


<a id="q1"></a>
### Q1. Filter and Add a Column

🟢 Warm-up · *Polars · filter, with\_columns*

From the orders, keep completed ones with amount ≥ `min_amount`, add `with_tax = round(amount × 1.18, 2)`, and return rows `(id, amount, with_tax)` sorted by id.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}], min_amount = 300` → `[(1, 1200.0, 1416.0), (4, 640.0, 755.2), (5, 2100.0, 2478.0), (6, 330.0, 389.4)]`

In [ ]:
def big_orders(rows, min_amount):
    """
    From the orders, keep completed ones with amount ≥ min_amount, add with_tax
    = round(amount × 1.18, 2), and return rows (id, amount, with_tax) sorted by
    id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_orders, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}], 300), [(1, 1200.0, 1416.0), (4, 640.0, 755.2), (5, 2100.0, 2478.0), (6, 330.0, 389.4)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`filter`, `with_columns`, `select`, `sort`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A loop with if-statements.

```python
def big_orders(rows, min_amount):
    out = []
    for r in rows:
        if r["status"] == "completed" and r["amount"] >= min_amount:
            out.append((r["id"], r["amount"], round(r["amount"] * 1.18, 2)))
    return sorted(out)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `filter`, `with_columns`, `select`, `sort`.

```python
def big_orders(rows, min_amount):
    df = pl.DataFrame(rows)
    return (df.filter((pl.col("status") == "completed") & (pl.col("amount") >= min_amount))
              .with_columns((pl.col("amount") * 1.18).round(2).alias("with_tax"))
              .select("id", "amount", "with_tax").sort("id").rows())
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with if-statements. That costs O(n log n) time and O(n) space."

"I can do better: `filter`, `with_columns`, `select`, `sort`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/filter-and-add-a-column)

<a id="q2"></a>
### Q2. Bucket Values With when/then

🟢 Warm-up · *Polars · conditional columns*

Label each order's amount as `'small'` (&lt; 200), `'medium'` (&lt; 1000) or `'large'`, and return the count of orders per band as `(band, count)` sorted by band.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}]` → `[('large', 2), ('medium', 4), ('small', 2)]`

In [ ]:
def amount_bands(rows):
    """
    Label each order's amount as 'small' (< 200), 'medium' (< 1000) or 'large',
    and return the count of orders per band as (band, count) sorted by band.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(amount_bands, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}],), [('large', 2), ('medium', 4), ('small', 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

if/elif per row and a counting dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pl.when().then().otherwise()` then `group_by`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** if/elif per row and a counting dict.

```python
def amount_bands(rows):
    counts = {}
    for r in rows:
        a = r["amount"]
        band = "small" if a < 200 else "medium" if a < 1000 else "large"
        counts[band] = counts.get(band, 0) + 1
    return sorted(counts.items())
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pl.when().then().otherwise()` then `group_by`.

```python
def amount_bands(rows):
    a = pl.col("amount")
    band = pl.when(a < 200).then(pl.lit("small")).when(a < 1000).then(pl.lit("medium")).otherwise(pl.lit("large"))
    return pl.DataFrame(rows).group_by(band.alias("band")).agg(pl.len()).sort("band").rows()
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: if/elif per row and a counting dict. That costs O(n) time and O(1) space."

"I can do better: `pl.when().then().otherwise()` then `group_by`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/bucket-values-with-when-then)

<a id="q3"></a>
### Q3. Group and Aggregate

🟢 Warm-up · *Polars · group\_by().agg()*

For completed orders, return per city `(city, orders, revenue, average order rounded to 1)` sorted by revenue, highest first.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}]` → `[('Goa', 1, 2100.0, 2100.0), ('Pune', 4, 2070.0, 517.5), ('Delhi', 1, 330.0, 330.0)]`

In [ ]:
def city_summary(rows):
    """
    For completed orders, return per city (city, orders, revenue, average order
    rounded to 1) sorted by revenue, highest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_summary, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}],), [('Goa', 1, 2100.0, 2100.0), ('Pune', 4, 2070.0, 517.5), ('Delhi', 1, 330.0, 330.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict of [count, total] lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`group_by('city').agg(...)` with several expressions.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A dict of [count, total] lists.

```python
def city_summary(rows):
    s = {}
    for r in rows:
        if r["status"] == "completed":
            s.setdefault(r["city"], [0, 0.0])
            s[r["city"]][0] += 1
            s[r["city"]][1] += r["amount"]
    out = [(c, n, t, round(t / n, 1)) for c, (n, t) in s.items()]
    return sorted(out, key=lambda x: -x[2])
```

**Cost:** O(n log c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `group_by('city').agg(...)` with several expressions.

```python
def city_summary(rows):
    return (pl.DataFrame(rows).filter(pl.col("status") == "completed")
              .group_by("city")
              .agg(pl.len().alias("orders"), pl.col("amount").sum().alias("revenue"), pl.col("amount").mean().round(1).alias("avg"))
              .sort("revenue", descending=True).rows())
```

**Cost:** O(n log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict of [count, total] lists. That costs O(n log c) time and O(c) space."

"I can do better: `group_by('city').agg(...)` with several expressions. That's O(n log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/group-and-aggregate)

<a id="q4"></a>
### Q4. A Rolling Average

🟢 Warm-up · *Polars · rolling\_mean*

Return each day's orders and the trailing 3-day average (rounded to 2; None for the first two days, which don't have 3 days yet).

**Examples**

- `rows = [{'day': '2026-11-01', 'orders': 12}, {'day': '2026-11-02', 'orders': 15}, {'day': '2026-11-03', 'orders': 9}, {'day': '2026-11-04', 'orders': 22}, {'day': '2026-11-05', 'orders': 30}, {'day': '2026-11-06', 'orders': 28}, {'day': '2026-11-07', 'orders': 11}, {'day': '2026-11-08', 'orders': 14}, {'day': '2026-11-09', 'orders': 19}, {'day': '2026-11-10', 'orders': 25}]` → `[('2026-11-01', 12, None), ('2026-11-02', 15, None), ('2026-11-03', 9, 12.0), ('2026-11-04', 22, 15.33), ('2026-11-05', 30, 20.33), ('2026-11-06', 28, 26.67), ('2026-11-07', 11, 23.0), ('2026-11-08', 14, 17.67), ('2026-11-09', 19, 14.67), ('2026-11-10', 25, 19.33)]`

In [ ]:
def rolling_avg(rows):
    """
    Return each day's orders and the trailing 3-day average (rounded to 2; None
    for the first two days, which don't have 3 days yet).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rolling_avg, [
    (([{'day': '2026-11-01', 'orders': 12}, {'day': '2026-11-02', 'orders': 15}, {'day': '2026-11-03', 'orders': 9}, {'day': '2026-11-04', 'orders': 22}, {'day': '2026-11-05', 'orders': 30}, {'day': '2026-11-06', 'orders': 28}, {'day': '2026-11-07', 'orders': 11}, {'day': '2026-11-08', 'orders': 14}, {'day': '2026-11-09', 'orders': 19}, {'day': '2026-11-10', 'orders': 25}],), [('2026-11-01', 12, None), ('2026-11-02', 15, None), ('2026-11-03', 9, 12.0), ('2026-11-04', 22, 15.33), ('2026-11-05', 30, 20.33), ('2026-11-06', 28, 26.67), ('2026-11-07', 11, 23.0), ('2026-11-08', 14, 17.67), ('2026-11-09', 19, 14.67), ('2026-11-10', 25, 19.33)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Slice the previous 3 values in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pl.col('orders').rolling_mean(window_size=3)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Slice the previous 3 values in a loop.

```python
def rolling_avg(rows):
    out = []
    for i, r in enumerate(rows):
        avg = None if i < 2 else round(sum(x["orders"] for x in rows[i - 2:i + 1]) / 3, 2)
        out.append((r["day"], r["orders"], avg))
    return out
```

**Cost:** O(n · w) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pl.col('orders').rolling_mean(window_size=3)`.

```python
def rolling_avg(rows):
    return pl.DataFrame(rows).with_columns(pl.col("orders").rolling_mean(window_size=3).round(2).alias("avg3")).rows()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Slice the previous 3 values in a loop. That costs O(n · w) time and O(n) space."

"I can do better: `pl.col('orders').rolling_mean(window_size=3)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/a-rolling-average)

<a id="q5"></a>
### Q5. Clean Strings and Extract Parts

🟢 Warm-up · *Polars · str namespace*

Strip spaces, lower-case each email, extract the domain after `@` with a regex (rows that don't match are dropped), and return `(domain, users)` sorted by users then domain.

**Examples**

- `rows = [{'email': 'Asha.K@Gmail.com'}, {'email': 'ravi@company.in'}, {'email': 'meera@gmail.com'}, {'email': ' dev@Outlook.com '}, {'email': 'kim@company.in'}, {'email': 'not-an-email'}, {'email': 'lee@gmail.com'}]` → `[('gmail.com', 3), ('company.in', 2), ('outlook.com', 1)]`

In [ ]:
import re

def email_domains(rows):
    """
    Strip spaces, lower-case each email, extract the domain after @ with a regex
    (rows that don't match are dropped), and return (domain, users) sorted by
    users then domain.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(email_domains, [
    (([{'email': 'Asha.K@Gmail.com'}, {'email': 'ravi@company.in'}, {'email': 'meera@gmail.com'}, {'email': ' dev@Outlook.com '}, {'email': 'kim@company.in'}, {'email': 'not-an-email'}, {'email': 'lee@gmail.com'}],), [('gmail.com', 3), ('company.in', 2), ('outlook.com', 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

String methods and a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`str.strip_chars`, `str.to_lowercase`, `str.extract`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** String methods and a dict.

```python
def email_domains(rows):
    counts = {}
    for r in rows:
        e = r["email"].strip().lower()
        m = re.fullmatch(r"[^@\s]+@([^@\s]+)", e)
        if m:
            counts[m.group(1)] = counts.get(m.group(1), 0) + 1
    return sorted(counts.items(), key=lambda x: (-x[1], x[0]))
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `str.strip_chars`, `str.to_lowercase`, `str.extract`.

```python
def email_domains(rows):
    return (pl.DataFrame(rows)
              .with_columns(pl.col("email").str.strip_chars().str.to_lowercase().str.extract(r"^[^@\s]+@([^@\s]+)$", 1).alias("domain"))
              .drop_nulls("domain").group_by("domain").agg(pl.len().alias("users"))
              .sort(["users", "domain"], descending=[True, False]).rows())
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: String methods and a dict. That costs O(n) time and O(d) space."

"I can do better: `str.strip_chars`, `str.to_lowercase`, `str.extract`. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/clean-strings-and-extract-parts)

<a id="q6"></a>
### Q6. Explode Lists Into Rows

🟢 Warm-up · *Polars · explode*

Each post has a list of tags. Return `(tag, posts)` counts, most used first (ties alphabetical). `explode` turns one row per post into one row per tag.

**Examples**

- `rows = [{'post': 1, 'tags': ['python', 'pandas']}, {'post': 2, 'tags': ['python', 'polars', 'speed']}, {'post': 3, 'tags': []}, {'post': 4, 'tags': ['sql', 'duckdb', 'python']}, {'post': 5, 'tags': ['polars']}]` → `[('python', 3), ('polars', 2), ('duckdb', 1), ('pandas', 1), ('speed', 1), ('sql', 1)]`

In [ ]:
def tag_counts(rows):
    """
    Each post has a list of tags. Return (tag, posts) counts, most used first
    (ties alphabetical). explode turns one row per post into one row per tag.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tag_counts, [
    (([{'post': 1, 'tags': ['python', 'pandas']}, {'post': 2, 'tags': ['python', 'polars', 'speed']}, {'post': 3, 'tags': []}, {'post': 4, 'tags': ['sql', 'duckdb', 'python']}, {'post': 5, 'tags': ['polars']}],), [('python', 3), ('polars', 2), ('duckdb', 1), ('pandas', 1), ('speed', 1), ('sql', 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops and a Counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`explode('tags')`, then `group_by`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested loops and a Counter.

```python
def tag_counts(rows):
    c = {}
    for r in rows:
        for t in r["tags"]:
            c[t] = c.get(t, 0) + 1
    return sorted(c.items(), key=lambda x: (-x[1], x[0]))
```

**Cost:** O(t) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `explode('tags')`, then `group_by`.

```python
def tag_counts(rows):
    return (pl.DataFrame(rows).explode("tags").drop_nulls("tags")
              .group_by("tags").agg(pl.len().alias("posts"))
              .sort(["posts", "tags"], descending=[True, False]).rows())
```

**Cost:** O(t) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops and a Counter. That costs O(t) time and O(u) space."

"I can do better: `explode('tags')`, then `group_by`. That's O(t) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/explode-lists-into-rows)

<a id="q7"></a>
### Q7. Keep the Latest Version of Each Row

🟢 Warm-up · *Polars · sort + unique*

A change feed has several versions per id. Keep each id's row with the latest `updated` date and return `(id, email)` sorted by id.

**Examples**

- `rows = [{'id': 1, 'email': 'a@x.in', 'updated': '2026-11-01'}, {'id': 2, 'email': 'b@x.in', 'updated': '2026-11-02'}, {'id': 1, 'email': 'asha@x.in', 'updated': '2026-11-05'}, {'id': 3, 'email': 'c@x.in', 'updated': '2026-11-03'}, {'id': 2, 'email': 'b@x.in', 'updated': '2026-11-02'}, {'id': 1, 'email': 'asha.k@x.in', 'updated': '2026-11-04'}]` → `[(1, 'asha@x.in'), (2, 'b@x.in'), (3, 'c@x.in')]`

In [ ]:
def latest_rows(rows):
    """
    A change feed has several versions per id. Keep each id's row with the
    latest updated date and return (id, email) sorted by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(latest_rows, [
    (([{'id': 1, 'email': 'a@x.in', 'updated': '2026-11-01'}, {'id': 2, 'email': 'b@x.in', 'updated': '2026-11-02'}, {'id': 1, 'email': 'asha@x.in', 'updated': '2026-11-05'}, {'id': 3, 'email': 'c@x.in', 'updated': '2026-11-03'}, {'id': 2, 'email': 'b@x.in', 'updated': '2026-11-02'}, {'id': 1, 'email': 'asha.k@x.in', 'updated': '2026-11-04'}],), [(1, 'asha@x.in'), (2, 'b@x.in'), (3, 'c@x.in')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict of the newest row per id.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sort('updated').unique('id', keep='last')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A dict of the newest row per id.

```python
def latest_rows(rows):
    best = {}
    for r in rows:
        if r["id"] not in best or r["updated"] > best[r["id"]]["updated"]:
            best[r["id"]] = r
    return sorted((i, r["email"]) for i, r in best.items())
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `sort('updated').unique('id', keep='last')`.

```python
def latest_rows(rows):
    return (pl.DataFrame(rows).sort("updated").unique("id", keep="last", maintain_order=True)
              .select("id", "email").sort("id").rows())
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict of the newest row per id. That costs O(n) time and O(k) space."

"I can do better: `sort('updated').unique('id', keep='last')`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/keep-the-latest-version-of-each-row)

<a id="q8"></a>
### Q8. SQL on Your Data With HAVING

🟢 Warm-up · *DuckDB · GROUP BY … HAVING*

With DuckDB, find users who spent more than `min_total` in total across at least `min_orders` purchases. Return `(user, purchases, total)` sorted by total, highest first.

**Examples**

- `rows = [{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}], min_total = 60.0, min_orders = 2` → `[('u1', 3, 125.0), ('u3', 2, 100.0)]`

In [ ]:
def big_spenders(rows, min_total, min_orders):
    """
    With DuckDB, find users who spent more than min_total in total across at
    least min_orders purchases. Return (user, purchases, total) sorted by total,
    highest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_spenders, [
    (([{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}], 60.0, 2), [('u1', 3, 125.0), ('u3', 2, 100.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Dicts of counts and totals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One SQL query with HAVING.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Dicts of counts and totals.

```python
def big_spenders(rows, min_total, min_orders):
    s = {}
    for r in rows:
        s.setdefault(r["user"], [0, 0.0])
        s[r["user"]][0] += 1
        s[r["user"]][1] += r["amount"]
    out = [(u, n, t) for u, (n, t) in s.items() if t > min_total and n >= min_orders]
    return sorted(out, key=lambda x: -x[2])
```

**Cost:** O(n log u) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** One SQL query with HAVING.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def big_spenders(rows, min_total, min_orders):
    con = duckdb.connect()
    sql_table(con, "events", rows)
    return con.execute("""
        SELECT "user", COUNT(*) AS purchases, SUM(amount) AS total
        FROM events GROUP BY "user"
        HAVING SUM(amount) > ? AND COUNT(*) >= ?
        ORDER BY total DESC""", [min_total, min_orders]).fetchall()
```

**Cost:** O(n log u) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Dicts of counts and totals. That costs O(n log u) time and O(u) space."

"I can do better: One SQL query with HAVING. That's O(n log u) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/sql-on-your-data-with-having)

<a id="q9"></a>
### Q9. Conditional Counts With FILTER

🟢 Warm-up · *DuckDB · aggregate FILTER*

Using the orders, return per city `(city, completed, cancelled_or_returned, completion % rounded to 1)` sorted by city, with `COUNT(*) FILTER (WHERE …)`.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}]` → `[('Delhi', 1, 2, 33.3), ('Goa', 1, 0, 100.0), ('Pune', 4, 0, 100.0)]`

In [ ]:
def completion(rows):
    """
    Using the orders, return per city (city, completed, cancelled_or_returned,
    completion % rounded to 1) sorted by city, with COUNT(*) FILTER (WHERE …).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(completion, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}],), [('Delhi', 1, 2, 33.3), ('Goa', 1, 0, 100.0), ('Pune', 4, 0, 100.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two counting dicts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

FILTER clauses in one GROUP BY.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Two counting dicts.

```python
def completion(rows):
    s = {}
    for r in rows:
        s.setdefault(r["city"], [0, 0])
        if r["status"] == "completed":
            s[r["city"]][0] += 1
        else:
            s[r["city"]][1] += 1
    return [(c, a, b, round(a / (a + b) * 100, 1)) for c, (a, b) in sorted(s.items())]
```

**Cost:** O(n log c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** FILTER clauses in one GROUP BY.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def completion(rows):
    con = duckdb.connect()
    sql_table(con, "orders", rows)
    return con.execute("""
        SELECT city,
               COUNT(*) FILTER (WHERE status = 'completed') AS completed,
               COUNT(*) FILTER (WHERE status <> 'completed') AS other,
               ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'completed') / COUNT(*), 1) AS pct
        FROM orders GROUP BY city ORDER BY city""").fetchall()
```

**Cost:** O(n log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two counting dicts. That costs O(n log c) time and O(c) space."

"I can do better: FILTER clauses in one GROUP BY. That's O(n log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/conditional-counts-with-filter)

<a id="q10"></a>
### Q10. Collect Values Into Lists

🟢 Warm-up · *DuckDB · list and string aggregates*

Return per user `(user, days they bought as a sorted list, amounts joined into one string like '40.0|55.0|30.0' in date order)`, sorted by user — with `list(… ORDER BY …)` and `string_agg`.

**Examples**

- `rows = [{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}]` → `[('u1', ['2026-01-03', '2026-02-01', '2026-03-05'], '40.0|55.0|30.0'), ('u2', ['2026-01-10', '2026-02-20', '2026-03-28'], '15.0|25.0|5.0'), ('u3', ['2026-02-14', '2026-03-09'], '90.0|10.0'), ('u4', ['2026-03-21'], '70.0')]`

In [ ]:
def user_lists(rows):
    """
    Return per user (user, days they bought as a sorted list, amounts joined
    into one string like '40.0|55.0|30.0' in date order), sorted by user — with
    list(… ORDER BY …) and string_agg.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(user_lists, [
    (([{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}],), [('u1', ['2026-01-03', '2026-02-01', '2026-03-05'], '40.0|55.0|30.0'), ('u2', ['2026-01-10', '2026-02-20', '2026-03-28'], '15.0|25.0|5.0'), ('u3', ['2026-02-14', '2026-03-09'], '90.0|10.0'), ('u4', ['2026-03-21'], '70.0')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict of lists, sorted afterwards.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`list(day ORDER BY day)` and `string_agg(…, '|' ORDER BY day)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A dict of lists, sorted afterwards.

```python
def user_lists(rows):
    g = {}
    for r in sorted(rows, key=lambda r: r["day"]):
        g.setdefault(r["user"], []).append(r)
    return [(u, [r["day"] for r in rs], "|".join(str(r["amount"]) for r in rs)) for u, rs in sorted(g.items())]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `list(day ORDER BY day)` and `string_agg(…, '|' ORDER BY day)`.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def user_lists(rows):
    con = duckdb.connect()
    sql_table(con, "events", rows)
    return con.execute("""
        SELECT "user", list(day ORDER BY day), string_agg(CAST(amount AS VARCHAR), '|' ORDER BY day)
        FROM events GROUP BY "user" ORDER BY "user" """).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict of lists, sorted afterwards. That costs O(n log n) time and O(n) space."

"I can do better: `list(day ORDER BY day)` and `string_agg(…, '|' ORDER BY day)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/collect-values-into-lists)

<a id="q11"></a>
### Q11. Run SQL on a pandas DataFrame

🟢 Warm-up · *DuckDB · query DataFrames in place*

DuckDB can query a pandas DataFrame directly, without copying it into a database. Build a DataFrame from the orders, register it, and return each customer's completed revenue and share of the total (% rounded to 1) with `SUM(...) OVER ()`, as `(customer, revenue, share)` sorted by revenue, highest first.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}]` → `[('dev', 2100.0, 46.7), ('asha', 1840.0, 40.9), ('ravi', 330.0, 7.3), ('meera', 230.0, 5.1)]`

In [ ]:
def revenue_share(rows):
    """
    DuckDB can query a pandas DataFrame directly, without copying it into a
    database. Build a DataFrame from the orders, register it, and return each
    customer's completed revenue and share of the total (% rounded to 1) with
    SUM(...) OVER (), as (customer, revenue, share) sorted by revenue, highest
    first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(revenue_share, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}],), [('dev', 2100.0, 46.7), ('asha', 1840.0, 40.9), ('ravi', 330.0, 7.3), ('meera', 230.0, 5.1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two passes with a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`con.register('orders', df)` and a window over the whole result.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Two passes with a dict.

```python
def revenue_share(rows):
    rev = {}
    for r in rows:
        if r["status"] == "completed":
            rev[r["customer"]] = rev.get(r["customer"], 0.0) + r["amount"]
    total = sum(rev.values())
    return sorted(((c, v, round(v / total * 100, 1)) for c, v in rev.items()), key=lambda x: (-x[1], x[0]))
```

**Cost:** O(n + c log c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `con.register('orders', df)` and a window over the whole result.

```python
def revenue_share(rows):
    df = pd.DataFrame(rows)
    con = duckdb.connect()
    con.register("orders", df)
    return con.execute("""
        SELECT customer, SUM(amount) AS revenue, ROUND(100 * SUM(amount) / SUM(SUM(amount)) OVER (), 1) AS share
        FROM orders WHERE status = 'completed'
        GROUP BY customer ORDER BY revenue DESC, customer""").fetchall()
```

**Cost:** O(n + c log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two passes with a dict. That costs O(n + c log c) time and O(c) space."

"I can do better: `con.register('orders', df)` and a window over the whole result. That's O(n + c log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/run-sql-on-a-pandas-dataframe)

<a id="q12"></a>
### Q12. A Lazy Query

🟡 Interview standard · *Polars · lazy() and collect()*

Write the top-`k` customers by completed revenue as a **lazy** query: Polars builds a plan, pushes the filter down, and only computes on `collect()`. Return `(customer, revenue)` pairs, ties by name.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}], k = 3` → `[('dev', 2100.0), ('asha', 1840.0), ('ravi', 330.0)]`

In [ ]:
def top_customers(rows, k):
    """
    Write the top-k customers by completed revenue as a lazy query: Polars
    builds a plan, pushes the filter down, and only computes on collect().
    Return (customer, revenue) pairs, ties by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_customers, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}], 3), [('dev', 2100.0), ('asha', 1840.0), ('ravi', 330.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum per customer in a dict, sort, slice.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.lazy()…head(k).collect()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Sum per customer in a dict, sort, slice.

```python
def top_customers(rows, k):
    rev = {}
    for r in rows:
        if r["status"] == "completed":
            rev[r["customer"]] = rev.get(r["customer"], 0.0) + r["amount"]
    return sorted(rev.items(), key=lambda x: (-x[1], x[0]))[:k]
```

**Cost:** O(n + c log c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.lazy()…head(k).collect()`.

```python
def top_customers(rows, k):
    plan = (pl.DataFrame(rows).lazy()
              .filter(pl.col("status") == "completed")
              .group_by("customer").agg(pl.col("amount").sum().alias("revenue"))
              .sort(["revenue", "customer"], descending=[True, False])
              .head(k))
    return plan.collect().rows()
```

**Cost:** O(n + c log c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum per customer in a dict, sort, slice. That costs O(n + c log c) time and O(c) space."

"I can do better: `df.lazy()…head(k).collect()`. That's O(n + c log c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/a-lazy-query)

<a id="q13"></a>
### Q13. Rank Within Groups

🟡 Interview standard · *Polars · window expressions with over()*

For completed orders, add each order's rank by amount within its city (1 = largest, dense) and its share of the city's revenue (% rounded to 1). Return `(id, city, rank, share)` sorted by city then rank.

**Examples**

- `rows = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}]` → `[(6, 'Delhi', 1, 100.0), (5, 'Goa', 1, 100.0), (1, 'Pune', 1, 58.0), (4, 'Pune', 2, 30.9), (8, 'Pune', 3, 7.2), (3, 'Pune', 4, 3.9)]`

In [ ]:
def rank_in_city(rows):
    """
    For completed orders, add each order's rank by amount within its city (1 =
    largest, dense) and its share of the city's revenue (% rounded to 1). Return
    (id, city, rank, share) sorted by city then rank.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rank_in_city, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}],), [(6, 'Delhi', 1, 100.0), (5, 'Goa', 1, 100.0), (1, 'Pune', 1, 58.0), (4, 'Pune', 2, 30.9), (8, 'Pune', 3, 7.2), (3, 'Pune', 4, 3.9)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group lists, sort each, compute shares.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`rank(...).over('city')` and `sum().over('city')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Group lists, sort each, compute shares.

```python
def rank_in_city(rows):
    groups = {}
    for r in rows:
        if r["status"] == "completed":
            groups.setdefault(r["city"], []).append(r)
    out = []
    for city, rs in groups.items():
        total = sum(r["amount"] for r in rs)
        distinct = sorted({r["amount"] for r in rs}, reverse=True)
        for r in rs:
            out.append((r["id"], city, distinct.index(r["amount"]) + 1, round(r["amount"] / total * 100, 1)))
    return sorted(out, key=lambda x: (x[1], x[2], x[0]))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `rank(...).over('city')` and `sum().over('city')`.

```python
def rank_in_city(rows):
    return (pl.DataFrame(rows).filter(pl.col("status") == "completed")
              .with_columns(pl.col("amount").rank("dense", descending=True).over("city").cast(pl.Int64).alias("rank"),
                            (pl.col("amount") / pl.col("amount").sum().over("city") * 100).round(1).alias("share"))
              .select("id", "city", "rank", "share").sort(["city", "rank", "id"]).rows())
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group lists, sort each, compute shares. That costs O(n log n) time and O(n) space."

"I can do better: `rank(...).over('city')` and `sum().over('city')`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/rank-within-groups)

<a id="q14"></a>
### Q14. Left Joins and Anti Joins

🟡 Interview standard · *Polars · join(how=...)*

Return `(completed revenue per segment sorted by segment, customers with no orders at all sorted)`. Revenue per segment needs a join from orders to customers; customers without orders need an **anti join**.

**Examples**

- `orders = [{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}], customers = [{'customer': 'asha', 'segment': 'pro'}, {'customer': 'ravi', 'segment': 'free'}, {'customer': 'meera', 'segment': 'pro'}, {'customer': 'dev', 'segment': 'enterprise'}, {'customer': 'lee', 'segment': 'free'}, {'customer': 'kim', 'segment': 'free'}]` → `([('enterprise', 2100.0), ('free', 330.0), ('pro', 2070.0)], ['lee'])`

In [ ]:
def segment_join(orders, customers):
    """
    Return (completed revenue per segment sorted by segment, customers with no
    orders at all sorted). Revenue per segment needs a join from orders to
    customers; customers without orders need an anti join.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(segment_join, [
    (([{'id': 1, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 1200.0}, {'id': 2, 'customer': 'ravi', 'city': 'Delhi', 'status': 'cancelled', 'amount': 450.0}, {'id': 3, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 80.0}, {'id': 4, 'customer': 'asha', 'city': 'Pune', 'status': 'completed', 'amount': 640.0}, {'id': 5, 'customer': 'dev', 'city': 'Goa', 'status': 'completed', 'amount': 2100.0}, {'id': 6, 'customer': 'ravi', 'city': 'Delhi', 'status': 'completed', 'amount': 330.0}, {'id': 7, 'customer': 'kim', 'city': 'Delhi', 'status': 'returned', 'amount': 990.0}, {'id': 8, 'customer': 'meera', 'city': 'Pune', 'status': 'completed', 'amount': 150.0}], [{'customer': 'asha', 'segment': 'pro'}, {'customer': 'ravi', 'segment': 'free'}, {'customer': 'meera', 'segment': 'pro'}, {'customer': 'dev', 'segment': 'enterprise'}, {'customer': 'lee', 'segment': 'free'}, {'customer': 'kim', 'segment': 'free'}]), ([('enterprise', 2100.0), ('free', 330.0), ('pro', 2070.0)], ['lee'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Dict lookups and set differences.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`join(how='left')` and `join(how='anti')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Dict lookups and set differences.

```python
def segment_join(orders, customers):
    seg = {c["customer"]: c["segment"] for c in customers}
    rev = {}
    for o in orders:
        if o["status"] == "completed":
            s = seg[o["customer"]]
            rev[s] = rev.get(s, 0.0) + o["amount"]
    buyers = {o["customer"] for o in orders}
    return (sorted(rev.items()), sorted(c["customer"] for c in customers if c["customer"] not in buyers))
```

**Cost:** O(n + c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `join(how='left')` and `join(how='anti')`.

```python
def segment_join(orders, customers):
    o, c = pl.DataFrame(orders), pl.DataFrame(customers)
    rev = (o.filter(pl.col("status") == "completed").join(c, on="customer", how="left")
            .group_by("segment").agg(pl.col("amount").sum()).sort("segment").rows())
    never = c.join(o, on="customer", how="anti").sort("customer")["customer"].to_list()
    return (rev, never)
```

**Cost:** O(n + c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Dict lookups and set differences. That costs O(n + c) time and O(c) space."

"I can do better: `join(how='left')` and `join(how='anti')`. That's O(n + c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/left-joins-and-anti-joins)

<a id="q15"></a>
### Q15. Pivot Long to Wide

🟡 Interview standard · *Polars · pivot*

Turn long sales rows into one row per month with a column per city (missing values → 0.0). Return the column names, then the rows, sorted by month; city columns sorted alphabetically.

**Examples**

- `rows = [{'month': '2026-01', 'city': 'Pune', 'revenue': 120.0}, {'month': '2026-01', 'city': 'Delhi', 'revenue': 90.0}, {'month': '2026-02', 'city': 'Pune', 'revenue': 150.0}, {'month': '2026-02', 'city': 'Goa', 'revenue': 40.0}, {'month': '2026-03', 'city': 'Delhi', 'revenue': 110.0}, {'month': '2026-03', 'city': 'Pune', 'revenue': 95.0}]` → `(['month', 'Delhi', 'Goa', 'Pune'], [('2026-01', 90.0, 0.0, 120.0), ('2026-02', 0.0, 40.0, 150.0), ('2026-03', 110.0, 0.0, 95.0)])`

In [ ]:
def pivot_sales(rows):
    """
    Turn long sales rows into one row per month with a column per city (missing
    values → 0.0). Return the column names, then the rows, sorted by month; city
    columns sorted alphabetically.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pivot_sales, [
    (([{'month': '2026-01', 'city': 'Pune', 'revenue': 120.0}, {'month': '2026-01', 'city': 'Delhi', 'revenue': 90.0}, {'month': '2026-02', 'city': 'Pune', 'revenue': 150.0}, {'month': '2026-02', 'city': 'Goa', 'revenue': 40.0}, {'month': '2026-03', 'city': 'Delhi', 'revenue': 110.0}, {'month': '2026-03', 'city': 'Pune', 'revenue': 95.0}],), (['month', 'Delhi', 'Goa', 'Pune'], [('2026-01', 90.0, 0.0, 120.0), ('2026-02', 0.0, 40.0, 150.0), ('2026-03', 110.0, 0.0, 95.0)])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dicts, then build rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pivot(on='city', index='month', values='revenue')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested dicts, then build rows.

```python
def pivot_sales(rows):
    cities = sorted({r["city"] for r in rows})
    table = {}
    for r in rows:
        table.setdefault(r["month"], {})[r["city"]] = r["revenue"]
    out = [tuple([m] + [table[m].get(c, 0.0) for c in cities]) for m in sorted(table)]
    return (["month"] + cities, out)
```

**Cost:** O(n + m · c) time, O(m · c) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pivot(on='city', index='month', values='revenue')`.

```python
def pivot_sales(rows):
    wide = pl.DataFrame(rows).pivot(on="city", index="month", values="revenue", aggregate_function="sum")
    cities = sorted(c for c in wide.columns if c != "month")
    wide = wide.select("month", *cities).fill_null(0.0).sort("month")
    return (wide.columns, wide.rows())
```

**Cost:** O(n + m · c) time, O(m · c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dicts, then build rows. That costs O(n + m · c) time and O(m · c) space."

"I can do better: `pivot(on='city', index='month', values='revenue')`. That's O(n + m · c) time and O(m · c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/pivot-long-to-wide)

<a id="q16"></a>
### Q16. Fill Gaps Per Group

🟡 Interview standard · *Polars · forward\_fill().over()*

Sensors miss readings. Forward-fill each sensor's temperature from its own previous reading (never from another sensor); a gap with no earlier reading stays None. Return `(nulls before, nulls after, filled rows)`.

**Examples**

- `rows = [{'sensor': 'a', 't': 1, 'temp': 21.0}, {'sensor': 'a', 't': 2, 'temp': None}, {'sensor': 'a', 't': 3, 'temp': None}, {'sensor': 'a', 't': 4, 'temp': 23.5}, {'sensor': 'b', 't': 1, 'temp': None}, {'sensor': 'b', 't': 2, 'temp': 18.0}, {'sensor': 'b', 't': 3, 'temp': None}]` → `(4, 1, [('a', 1, 21.0), ('a', 2, 21.0), ('a', 3, 21.0), ('a', 4, 23.5), ('b', 1, None), ('b', 2, 18.0), ('b', 3, 18.0)])`

In [ ]:
def fill_gaps(rows):
    """
    Sensors miss readings. Forward-fill each sensor's temperature from its own
    previous reading (never from another sensor); a gap with no earlier reading
    stays None. Return (nulls before, nulls after, filled rows).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fill_gaps, [
    (([{'sensor': 'a', 't': 1, 'temp': 21.0}, {'sensor': 'a', 't': 2, 'temp': None}, {'sensor': 'a', 't': 3, 'temp': None}, {'sensor': 'a', 't': 4, 'temp': 23.5}, {'sensor': 'b', 't': 1, 'temp': None}, {'sensor': 'b', 't': 2, 'temp': 18.0}, {'sensor': 'b', 't': 3, 'temp': None}],), (4, 1, [('a', 1, 21.0), ('a', 2, 21.0), ('a', 3, 21.0), ('a', 4, 23.5), ('b', 1, None), ('b', 2, 18.0), ('b', 3, 18.0)])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Remember the last value per sensor while looping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pl.col('temp').forward_fill().over('sensor')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Remember the last value per sensor while looping.

```python
def fill_gaps(rows):
    last, out = {}, []
    for r in sorted(rows, key=lambda r: (r["sensor"], r["t"])):
        v = r["temp"] if r["temp"] is not None else last.get(r["sensor"])
        if r["temp"] is not None:
            last[r["sensor"]] = r["temp"]
        out.append((r["sensor"], r["t"], v))
    before = sum(r["temp"] is None for r in rows)
    return (before, sum(v is None for *_, v in out), out)
```

**Cost:** O(n) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pl.col('temp').forward_fill().over('sensor')`.

```python
def fill_gaps(rows):
    df = pl.DataFrame(rows).sort("sensor", "t")
    filled = df.with_columns(pl.col("temp").forward_fill().over("sensor"))
    return (df["temp"].null_count(), filled["temp"].null_count(), filled.rows())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Remember the last value per sensor while looping. That costs O(n) time and O(s) space."

"I can do better: `pl.col('temp').forward_fill().over('sensor')`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/fill-gaps-per-group)

<a id="q17"></a>
### Q17. Running Totals and Month-Over-Month

🟡 Interview standard · *DuckDB · window functions, LAG*

Return per month `(month 'YYYY-MM', revenue, running total, growth % vs the previous month rounded to 1 — None for the first month)`.

**Examples**

- `rows = [{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}]` → `[('2026-01', 55.0, 55.0, None), ('2026-02', 170.0, 225.0, 209.1), ('2026-03', 115.0, 340.0, -32.4)]`

In [ ]:
def monthly_growth(rows):
    """
    Return per month (month 'YYYY-MM', revenue, running total, growth % vs the
    previous month rounded to 1 — None for the first month).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(monthly_growth, [
    (([{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}],), [('2026-01', 55.0, 55.0, None), ('2026-02', 170.0, 225.0, 209.1), ('2026-03', 115.0, 340.0, -32.4)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum per month, then walk the sorted months.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SUM() OVER (ORDER BY …)` and `LAG()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Sum per month, then walk the sorted months.

```python
def monthly_growth(rows):
    m = {}
    for r in rows:
        m[r["day"][:7]] = m.get(r["day"][:7], 0.0) + r["amount"]
    out, running, prev = [], 0.0, None
    for month in sorted(m):
        running += m[month]
        growth = None if prev is None else round((m[month] / prev - 1) * 100, 1)
        out.append((month, m[month], running, growth))
        prev = m[month]
    return out
```

**Cost:** O(n + m log m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `SUM() OVER (ORDER BY …)` and `LAG()`.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def monthly_growth(rows):
    con = duckdb.connect()
    sql_table(con, "events", rows)
    return con.execute("""
        WITH m AS (SELECT substr(day, 1, 7) AS month, SUM(amount) AS revenue FROM events GROUP BY 1)
        SELECT month, revenue,
               SUM(revenue) OVER (ORDER BY month) AS running,
               ROUND((revenue / LAG(revenue) OVER (ORDER BY month) - 1) * 100, 1) AS growth
        FROM m ORDER BY month""").fetchall()
```

**Cost:** O(n + m log m) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum per month, then walk the sorted months. That costs O(n + m log m) time and O(m) space."

"I can do better: `SUM() OVER (ORDER BY …)` and `LAG()`. That's O(n + m log m) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/running-totals-and-month-over-month)

<a id="q18"></a>
### Q18. Top Row per Group With QUALIFY

🟡 Interview standard · *DuckDB · QUALIFY*

For each user, return their single largest purchase `(user, day, amount)` (ties: earliest day), sorted by user. DuckDB's `QUALIFY` filters on a window function without a subquery.

**Examples**

- `rows = [{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}]` → `[('u1', '2026-02-01', 55.0), ('u2', '2026-02-20', 25.0), ('u3', '2026-02-14', 90.0), ('u4', '2026-03-21', 70.0)]`

In [ ]:
def biggest_purchase(rows):
    """
    For each user, return their single largest purchase (user, day, amount)
    (ties: earliest day), sorted by user. DuckDB's QUALIFY filters on a window
    function without a subquery.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(biggest_purchase, [
    (([{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}],), [('u1', '2026-02-01', 55.0), ('u2', '2026-02-20', 25.0), ('u3', '2026-02-14', 90.0), ('u4', '2026-03-21', 70.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep the best row per user in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`QUALIFY ROW_NUMBER() OVER (PARTITION BY … ORDER BY …) = 1`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep the best row per user in a dict.

```python
def biggest_purchase(rows):
    best = {}
    for r in rows:
        b = best.get(r["user"])
        if b is None or r["amount"] > b["amount"] or (r["amount"] == b["amount"] and r["day"] < b["day"]):
            best[r["user"]] = r
    return [(u, best[u]["day"], best[u]["amount"]) for u in sorted(best)]
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `QUALIFY ROW_NUMBER() OVER (PARTITION BY … ORDER BY …) = 1`.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def biggest_purchase(rows):
    con = duckdb.connect()
    sql_table(con, "events", rows)
    return con.execute("""
        SELECT "user", day, amount FROM events
        QUALIFY ROW_NUMBER() OVER (PARTITION BY "user" ORDER BY amount DESC, day) = 1
        ORDER BY "user" """).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep the best row per user in a dict. That costs O(n) time and O(u) space."

"I can do better: `QUALIFY ROW_NUMBER() OVER (PARTITION BY … ORDER BY …) = 1`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/top-row-per-group-with-qualify)

<a id="q19"></a>
### Q19. Match Each Trade to the Latest Quote

🟡 Interview standard · *DuckDB · ASOF JOIN*

For each trade, find the most recent quote for the same symbol at or before the trade time, and return `(t, symbol, qty, price, value = qty × price)` sorted by time; a trade before any quote is dropped. This 'as of' join is common for time-series and finance data.

**Examples**

- `trades = [{'t': 3, 'symbol': 'INFY', 'qty': 10}, {'t': 7, 'symbol': 'TCS', 'qty': 5}, {'t': 9, 'symbol': 'INFY', 'qty': 20}, {'t': 15, 'symbol': 'INFY', 'qty': 5}, {'t': 16, 'symbol': 'TCS', 'qty': 8}, {'t': 1, 'symbol': 'TCS', 'qty': 2}], quotes = [{'t': 0, 'symbol': 'INFY', 'price': 1500.0}, {'t': 5, 'symbol': 'INFY', 'price': 1510.0}, {'t': 12, 'symbol': 'INFY', 'price': 1495.0}, {'t': 2, 'symbol': 'TCS', 'price': 3900.0}, {'t': 10, 'symbol': 'TCS', 'price': 3925.0}]` → `[(3, 'INFY', 10, 1500.0, 15000.0), (7, 'TCS', 5, 3900.0, 19500.0), (9, 'INFY', 20, 1510.0, 30200.0), (15, 'INFY', 5, 1495.0, 7475.0), (16, 'TCS', 8, 3925.0, 31400.0)]`

In [ ]:
def asof_prices(trades, quotes):
    """
    For each trade, find the most recent quote for the same symbol at or before
    the trade time, and return (t, symbol, qty, price, value = qty × price)
    sorted by time; a trade before any quote is dropped. This 'as of' join is
    common for time-series and finance data.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(asof_prices, [
    (([{'t': 3, 'symbol': 'INFY', 'qty': 10}, {'t': 7, 'symbol': 'TCS', 'qty': 5}, {'t': 9, 'symbol': 'INFY', 'qty': 20}, {'t': 15, 'symbol': 'INFY', 'qty': 5}, {'t': 16, 'symbol': 'TCS', 'qty': 8}, {'t': 1, 'symbol': 'TCS', 'qty': 2}], [{'t': 0, 'symbol': 'INFY', 'price': 1500.0}, {'t': 5, 'symbol': 'INFY', 'price': 1510.0}, {'t': 12, 'symbol': 'INFY', 'price': 1495.0}, {'t': 2, 'symbol': 'TCS', 'price': 3900.0}, {'t': 10, 'symbol': 'TCS', 'price': 3925.0}]), [(3, 'INFY', 10, 1500.0, 15000.0), (7, 'TCS', 5, 3900.0, 19500.0), (9, 'INFY', 20, 1510.0, 30200.0), (15, 'INFY', 5, 1495.0, 7475.0), (16, 'TCS', 8, 3925.0, 31400.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each trade, scan the quotes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ASOF JOIN … ON symbol AND trades.t >= quotes.t`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** For each trade, scan the quotes.

```python
def asof_prices(trades, quotes):
    out = []
    for tr in trades:
        best = None
        for q in quotes:
            if q["symbol"] == tr["symbol"] and q["t"] <= tr["t"] and (best is None or q["t"] > best["t"]):
                best = q
        if best:
            out.append((tr["t"], tr["symbol"], tr["qty"], best["price"], tr["qty"] * best["price"]))
    return sorted(out)
```

**Cost:** O(t · q) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `ASOF JOIN … ON symbol AND trades.t >= quotes.t`.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def asof_prices(trades, quotes):
    con = duckdb.connect()
    sql_table(con, "trades", trades)
    sql_table(con, "quotes", quotes)
    return con.execute("""
        SELECT tr.t, tr.symbol, tr.qty, q.price, tr.qty * q.price AS value
        FROM trades tr ASOF JOIN quotes q ON tr.symbol = q.symbol AND tr.t >= q.t
        ORDER BY tr.t""").fetchall()
```

**Cost:** O((t + q) log q) time, O(q) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each trade, scan the quotes. That costs O(t · q) time and O(1) space."

"I can do better: `ASOF JOIN … ON symbol AND trades.t >= quotes.t`. That's O((t + q) log q) time and O(q) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/match-each-trade-to-the-latest-quote)

<a id="q20"></a>
### Q20. Fill Missing Days With Zero

🟡 Interview standard · *DuckDB · generate\_series + LEFT JOIN*

Purchases only exist on some days. Return `(day 'YYYY-MM-DD', purchases, revenue)` for **every** day from `start` to `end` inclusive, with 0 and 0.0 on days without purchases — charts and moving averages break on missing days.

**Examples**

- `rows = [{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}], start = '2026-03-03', end = '2026-03-10'` → `[('2026-03-03', 0, 0.0), ('2026-03-04', 0, 0.0), ('2026-03-05', 1, 30.0), ('2026-03-06', 0, 0.0), ('2026-03-07', 0, 0.0), ('2026-03-08', 0, 0.0), ('2026-03-09', 1, 10.0), ('2026-03-10', 0, 0.0)]`

In [ ]:
from datetime import date, timedelta

def daily_filled(rows, start, end):
    """
    Purchases only exist on some days. Return (day 'YYYY-MM-DD', purchases,
    revenue) for every day from start to end inclusive, with 0 and 0.0 on days
    without purchases — charts and moving averages break on missing days.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(daily_filled, [
    (([{'user': 'u1', 'day': '2026-01-03', 'amount': 40.0}, {'user': 'u2', 'day': '2026-01-10', 'amount': 15.0}, {'user': 'u1', 'day': '2026-02-01', 'amount': 55.0}, {'user': 'u3', 'day': '2026-02-14', 'amount': 90.0}, {'user': 'u2', 'day': '2026-02-20', 'amount': 25.0}, {'user': 'u1', 'day': '2026-03-05', 'amount': 30.0}, {'user': 'u3', 'day': '2026-03-09', 'amount': 10.0}, {'user': 'u4', 'day': '2026-03-21', 'amount': 70.0}, {'user': 'u2', 'day': '2026-03-28', 'amount': 5.0}], '2026-03-03', '2026-03-10'), [('2026-03-03', 0, 0.0), ('2026-03-04', 0, 0.0), ('2026-03-05', 1, 30.0), ('2026-03-06', 0, 0.0), ('2026-03-07', 0, 0.0), ('2026-03-08', 0, 0.0), ('2026-03-09', 1, 10.0), ('2026-03-10', 0, 0.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the dates with timedelta and look up a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`generate_series` of dates LEFT JOIN the daily totals.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Walk the dates with timedelta and look up a dict.

```python
def daily_filled(rows, start, end):
    totals = {}
    for r in rows:
        n, s = totals.get(r["day"], (0, 0.0))
        totals[r["day"]] = (n + 1, s + r["amount"])
    out, d = [], date.fromisoformat(start)
    while d <= date.fromisoformat(end):
        n, s = totals.get(d.isoformat(), (0, 0.0))
        out.append((d.isoformat(), n, s))
        d += timedelta(days=1)
    return out
```

**Cost:** O(d + n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `generate_series` of dates LEFT JOIN the daily totals.

```python
def sql_table(con, name, rows):
    """Create a DuckDB table from a list of dicts (types taken from the first non-null value of each column)."""
    cols = list(rows[0])
    def sql_type(c):
        v = next((r[c] for r in rows if r[c] is not None), "")
        return "BOOLEAN" if isinstance(v, bool) else "BIGINT" if isinstance(v, int) else "DOUBLE" if isinstance(v, float) else "VARCHAR"
    con.execute(f"CREATE TABLE {name} ({', '.join(f'{c} {sql_type(c)}' for c in cols)})")
    con.executemany(f"INSERT INTO {name} VALUES ({', '.join('?' * len(cols))})", [[r[c] for c in cols] for r in rows])


def daily_filled(rows, start, end):
    con = duckdb.connect()
    sql_table(con, "events", rows)
    return con.execute("""
        WITH days AS (SELECT CAST(d AS DATE) AS day
                      FROM generate_series(CAST(? AS DATE), CAST(? AS DATE), INTERVAL 1 DAY) AS g(d)),
             t AS (SELECT CAST(day AS DATE) AS day, COUNT(*) AS n, SUM(amount) AS s FROM events GROUP BY 1)
        SELECT CAST(days.day AS VARCHAR), COALESCE(t.n, 0), COALESCE(t.s, 0.0)
        FROM days LEFT JOIN t USING (day) ORDER BY days.day""", [start, end]).fetchall()
```

**Cost:** O(d + n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the dates with timedelta and look up a dict. That costs O(d + n) time and O(d) space."

"I can do better: `generate_series` of dates LEFT JOIN the daily totals. That's O(d + n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/polars-duckdb/fill-missing-days-with-zero)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. When would you choose Polars over pandas?</b></summary>

For larger data or heavy transformations where speed and memory matter: Polars is multi-threaded, lazy and doesn't copy as much. pandas still wins on ecosystem coverage and familiarity; many teams use both.

</details>

<details>
<summary><b>2. What is DuckDB good for?</b></summary>

Fast analytical SQL on local files and DataFrames without running a database server — exploring Parquet on a laptop, transformations in pipelines, and as a dbt engine for small to medium data.

</details>

<details>
<summary><b>3. What does lazy evaluation buy you?</b></summary>

The engine sees the whole query before running it, so it can push filters and column selection into the file reader, combine steps and skip work — reading only what's needed.

</details>

<details>
<summary><b>4. Is DuckDB a replacement for Postgres?</b></summary>

No. Postgres is for many concurrent transactional reads and writes; DuckDB is an embedded engine for analytical queries in one process. They complement each other — DuckDB can even read Postgres tables.

</details>

<details>
<summary><b>5. How do pandas, Polars and DuckDB share data?</b></summary>

Through Apache Arrow: `pl.from_pandas`, `.to_pandas()`, and DuckDB's `.df()`, `.pl()` and `.arrow()` results, usually without copying much.

</details>

<details>
<summary><b>6. What is an ASOF join?</b></summary>

A join that matches each row to the most recent row at or before its time in another table — e.g. each trade to the latest price quote — instead of requiring equal timestamps.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why Polars and DuckDB?
- [ ] I can explain: Polars expressions
- [ ] I can explain: Reshaping and windows in Polars
- [ ] I can explain: DuckDB: SQL on files and DataFrames

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Fill Gaps Per Group
- [ ] [Q17](#q17) Running Totals and Month-Over-Month
- [ ] [Q18](#q18) Top Row per Group With QUALIFY
- [ ] [Q19](#q19) Match Each Trade to the Latest Quote
- [ ] [Q20](#q20) Fill Missing Days With Zero

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.